# Ćwiczenia 4 — Podział danych, potok i pierwszy model (wersja Python, z rozwiązaniami)
**Dane demo: kohorta T2D (QinJ_2012)**

Model uczenia maszynowego ma działać na próbkach, których nie widział podczas uczenia. Żeby to sprawdzić, część próbek odkłada się na bok jako zbiór testowy, a model uczy się wyłącznie na pozostałych, czyli na zbiorze treningowym. Na tym spotkaniu przechodzimy po raz pierwszy cały ten cykl: podział danych, przygotowanie danych na treningu, dopasowanie prostego modelu i ocena na teście. Na końcu sprawdzamy, jak bardzo wynik zależy od tego, które próbki trafiły do testu.

Notatnik najlepiej uruchamiać od początku po ponownym uruchomieniu jądra (*Kernel → Restart*), komórka po komórce: obiekty pozostawione w pamięci przez wcześniejszy kod mogą mieć te same nazwy i inną zawartość.

## Wczytanie danych

Wczytujemy te same pliki CSV co na Ćwiczeniach 2 i 3: `X` to cechy (próbki × taksony, liczebności względne w procentach), `y` to klasa każdej próbki. Metadane nie są dziś potrzebne. Notatnik trzeba zapisać w tym samym katalogu co notatniki z poprzednich ćwiczeń, czyli obok katalogu `dane/`.

Jeśli `import sklearn` zgłasza błąd `No module named ...`, bibliotekę instaluje się jednorazowo w komórce notatnika, a potem *Kernel → Restart*:

```
%pip install scikit-learn
```

Instaluje się `scikit-learn`, a importuje `sklearn`; pakiet o nazwie `sklearn` to co innego (wycofany) i z nim kod nie działa.

In [ ]:
import numpy as np                       # obliczenia na tablicach
import pandas as pd                      # tabele danych
import matplotlib.pyplot as plt          # wykresy
from sklearn.model_selection import train_test_split   # podział na trening i test
from sklearn.model_selection import StratifiedShuffleSplit   # wiele podziałów warstwowych naraz
from sklearn.pipeline import make_pipeline             # potok: kroki i model w jednym obiekcie
from sklearn.preprocessing import StandardScaler       # standaryzacja
from sklearn.dummy import DummyClassifier              # klasyfikator bazowy
from sklearn.neighbors import KNeighborsClassifier     # k najbliższych sąsiadów
pd.set_option("display.width", 120)      # szersze wydruki tabel (bez zawijania kolumn)

X = pd.read_csv("dane/t2d_features.csv", index_col=0)   # index_col=0: pierwsza kolumna to nazwy próbek
# ["grupa"]: kolumna z etykietami; index=X.index: te same nazwy wierszy co w X
y = pd.Series(pd.read_csv("dane/t2d_labels.csv")["grupa"].to_numpy(), index=X.index)

print(X.shape)                           # (liczba próbek, liczba cech)
print(y.value_counts())                  # liczba próbek w każdej klasie

## 1. Podział na zbiór treningowy i testowy

Zbiór testowy odkłada się przed jakimkolwiek przygotowaniem danych i używa się go tylko do oceny gotowego modelu. Dzięki temu ocena odpowiada sytuacji, w której model dostaje nowe próbki. Podział jest **warstwowy** (ang. *stratified split*): próbki losuje się osobno w każdej klasie, więc udziały klas w treningu i w teście są takie same jak w całym zbiorze. Bez tego test mógłby przez przypadek zawierać na przykład wyraźnie więcej próbek T2D, a dokładność zmierzona na takim teście dotyczyłaby innej proporcji klas niż w danych. Do treningu trafia 75% próbek, do testu 25%.

Podział jest losowy. **Ziarno** (ang. *seed*) generatora liczb losowych ustala, które próbki zostaną wylosowane: to samo ziarno daje przy każdym uruchomieniu ten sam podział. W całym kursie używamy ziarna 42. R i Python mają różne generatory liczb losowych, więc dzielą dane inaczej i część liczb różni się między wersjami arkusza.

In [ ]:
# test_size=0.25: 25% próbek w teście; stratify=y: podział warstwowy; random_state=42: ziarno
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print({"train": len(y_tr), "test": len(y_te)})        # liczba próbek w każdym zbiorze
# udziały klas w całym zbiorze, w treningu i w teście; normalize=True: udziały zamiast liczebności
print(pd.DataFrame({"all": y.value_counts(normalize=True),
                    "train": y_tr.value_counts(normalize=True),
                    "test": y_te.value_counts(normalize=True)}).T[["control", "T2D"]].round(3))   # kolumny w kolejności control, T2D

**Na co patrzeć.** Trzy wiersze tabeli są prawie takie same: trening i test mają tę samą strukturę klas co cały zbiór. Przy podziale warstwowym tak jest przy każdym ziarnie: `train_test_split(..., stratify=y)` losuje próbki testowe osobno w każdej klasie, więc w teście zawsze są 43 próbki control i 43 próbki T2D (przy innym ziarnie liczby mogą się zmienić najwyżej o jedną próbkę przez zaokrąglenie).

Dla porównania wykonujemy podział **zwykły**: losujemy 86 próbek testowych bez względu na klasę, czyli bez argumentu `stratify`.

In [ ]:
# zwykły podział: bez stratify; _ to nieużywane etykiety treningu, y_r to etykiety testu
_, y_r = train_test_split(y, test_size=0.25, random_state=1)
print(y_r.value_counts())                # liczba próbek każdej klasy w takim teście
print(round((y_r == "T2D").mean(), 3))   # odsetek T2D w teście; średnia z True/False to odsetek True

Przy innym ziarnie wynik byłby inny. Żeby sprawdzić, jak bardzo, powtarzamy to losowanie 200 razy w pętli `for`. Wcięte linijki pod `for i in range(200):` wykonują się 200 razy, za każdym razem z inną wartością `i` (0, 1, …, 199), którą podajemy jako ziarno. Wynik każdego powtórzenia dopisujemy na koniec listy metodą `.append()`.

In [ ]:
share_random = []                        # pusta lista na wyniki
for i in range(200):                     # pętla: i przyjmuje kolejno wartości 0, 1, ..., 199
    _, y_r = train_test_split(y, test_size=0.25, random_state=i)   # zwykły podział z ziarnem i
    share_random.append((y_r == "T2D").mean())   # dopisanie odsetka T2D na koniec listy
share_random = np.array(share_random)    # lista → tablica NumPy, na której łatwo liczyć
print(len(share_random))                 # 200 wyników, po jednym na podział
print(share_random.min().round(3), share_random.max().round(3))   # najmniejszy i największy odsetek T2D w teście

# histogram 200 odsetków; linia przerywana = odsetek T2D w teście przy podziale warstwowym
fig, ax = plt.subplots(figsize=(7, 3.5))   # nowa rycina z jednym panelem; figsize: szerokość i wysokość w calach
ax.hist(share_random, bins=25, color="lightgrey", edgecolor="dimgrey")   # słupki: liczba podziałów
ax.axvline((y_te == "T2D").mean(), linestyle="--", color="#2C5F7C")    # podział warstwowy
ax.set_xlabel("Odsetek próbek T2D w zbiorze testowym (podział zwykły)"); ax.set_ylabel("Liczba podziałów")   # podpisy osi
plt.show()   # wyświetlenie ryciny

**Na co patrzeć.** Linia przerywana to odsetek T2D w teście przy podziale warstwowym; przy każdym ziarnie wynosi praktycznie tyle samo. Przy zwykłym podziale odsetek zmienia się od podziału do podziału, czasem wyraźnie.

> **✏️ Zadanie 1.** Policzyć, w ilu z 200 zwykłych podziałów odsetek T2D w teście jest mniejszy niż 0,40 (w całym zbiorze wynosi 0,494). Jaką dokładność miałby na takim teście model, który zawsze wskazuje control?

In [ ]:
print((share_random < 0.40).sum())      # liczba zwykłych podziałów z odsetkiem T2D w teście poniżej 0,40
# klasyfikator bazowy zawsze wskazuje control, więc trafia tyle, ile jest w teście próbek control:
print((1 - share_random[share_random < 0.40]).round(3))   # jego dokładność na każdym z tych testów

**Rozwiązanie.** W 5 z 200 zwykłych podziałów test ma mniej niż 40% próbek T2D. Model, który
zawsze wskazuje control, trafia wtedy w każdą próbkę control, więc jego dokładność to 1 minus
odsetek T2D: na tych pięciu testach od 0,605 do 0,628, zamiast około 0,50 przy podziale
warstwowym. Model, który niczego się nie nauczył, wyglądałby na takim teście na całkiem dobry.
Wynik na zwykłym podziale zależy więc częściowo od proporcji klas, która akurat trafiła do
testu; podział warstwowy usuwa to źródło zmienności.

## 2. Przygotowanie danych na treningu

Przygotowanie danych składa się z kroków, które mają **parametry** wyznaczane z danych: filtr cech ma listę zachowanych cech, standaryzacja ma średnią i odchylenie standardowe każdej cechy. Parametry wyznacza się wyłącznie na treningu, a do testu stosuje się je bez zmian. Gdyby wyznaczyć je na całym zbiorze, informacja o próbkach testowych przeszłaby do przygotowania danych, a ocena na teście byłaby zawyżona. Takie zjawisko nazywa się **przeciekiem danych** (ang. *data leakage*); omówimy je dokładnie na Ćwiczeniach 5. Ciąg kroków dopasowanych na treningu i stosowanych w tej samej kolejności do nowych próbek nazywa się **potokiem** (ang. *pipeline*).

Pierwszy krok to filtr częstości występowania z Ćwiczeń 2: zostają cechy obecne w co najmniej 10% próbek, tym razem liczonych tylko w treningu.

In [ ]:
prevalence_tr = (X_tr > 0).mean()        # częstość występowania każdej cechy w treningu
keep = prevalence_tr.index[prevalence_tr >= 0.10]   # parametr kroku: nazwy zachowanych cech
print(len(keep))                         # liczba zachowanych cech
X_tr_f = X_tr[keep]                      # trening: tylko zachowane cechy
X_te_f = X_te[keep]                      # test: te same cechy, bez sprawdzania wartości w teście

**Na co patrzeć.** Test dostaje dokładnie te kolumny, które wybrano na treningu, niezależnie od tego, jak często cechy występują w samym teście.

Drugi krok to przekształcenie CLR z Ćwiczeń 3: logarytm każdej wartości minus średnia logarytmów tej samej próbki, z dodaniem `1e-6` do zer. CLR liczy się w każdej próbce osobno, więc nie ma parametrów szacowanych z innych próbek. Zależy jednak od zestawu cech, dlatego stosuje się je po filtrze.

In [ ]:
# własna funkcja: CLR dla każdego wiersza (próbki) tabeli M
def clr(M):
    L = np.log(M + 1e-6)                 # logarytm naturalny; 1e-6 zastępuje zera
    return L.sub(L.mean(axis=1), axis=0) # od każdego wiersza odejmujemy jego średnią

Z_tr = clr(X_tr_f)                       # trening po CLR
Z_te = clr(X_te_f)                       # test po CLR
print(Z_tr.shape)                        # liczba próbek treningowych i cech

Trzeci krok to **standaryzacja** (ang. *standardization*): od każdej cechy odejmuje się jej średnią i dzieli się ją przez odchylenie standardowe, więc wszystkie cechy mają tę samą skalę. Na Ćwiczeniach 2 i 3 okazało się, że bez wspólnej skali odległości między próbkami wyznacza kilka cech o największym rozrzucie; dlatego przed modelem opartym na odległościach, użytym w sekcji 3, cechy zwykle się standaryzuje. `StandardScaler` wyznacza średnie i odchylenia metodą `.fit()` na treningu, a `.transform()` stosuje je do dowolnego zbioru.

In [ ]:
scaler = StandardScaler().fit(Z_tr)      # .fit(): średnie i odchylenia standardowe z treningu
S_tr = scaler.transform(Z_tr)            # trening po standaryzacji (tablica NumPy)
S_te = scaler.transform(Z_te)            # test po standaryzacji tymi samymi parametrami

print(S_tr.mean(axis=0).min().round(3) + 0, S_tr.mean(axis=0).max().round(3) + 0)   # średnie cech w treningu (+ 0 usuwa zapis „-0.0”)
print(S_te.mean(axis=0).min().round(3), S_te.mean(axis=0).max().round(3))   # średnie cech w teście

fig, ax = plt.subplots(figsize=(8, 3.5))   # nowa rycina z jednym panelem
ax.hist(S_te.mean(axis=0), bins=30, color="lightgrey", edgecolor="dimgrey")   # histogram średnich cech w teście
ax.axvline(0, linestyle="--", color="black")   # w treningu każda średnia jest równa 0
ax.set_xlabel("Średnia cechy w teście po standaryzacji"); ax.set_ylabel("Liczba cech")   # podpisy osi
plt.show()   # wyświetlenie ryciny

**Na co patrzeć.** W treningu średnie wszystkich cech są równe zeru, bo parametry policzono na tych samych próbkach. W teście są bliskie zeru, ale nie równe: test to inne próbki. Tak ma być, bo model w praktyce dostaje nowe próbki przekształcone parametrami z treningu.

🔧 **Do sprawdzenia.** Wyznaczyć standaryzację na samym teście (`StandardScaler().fit_transform(Z_te)`) i wypisać zakres średnich cech. Dlaczego są wtedy równe zeru i dlaczego tak się nie postępuje?

> **✏️ Zadanie 2.** Policzyć, ile cech z listy `keep` występuje w teście w mniej niż 10% próbek. Czy należy je usunąć z testu?

In [ ]:
# (X_te[keep] > 0).mean(): częstość występowania cech z listy keep, policzona w teście
print(((X_te[keep] > 0).mean() < 0.10).sum())   # ile z nich występuje w teście w mniej niż 10% próbek

**Rozwiązanie.** 12 ze 197 cech zachowanych na treningu występuje w teście w mniej niż
10% próbek; gdyby filtr policzyć na teście, te cechy by odpadły. Nie usuwa się ich jednak
z testu. Lista cech jest parametrem przygotowania danych, wyznaczonym na treningu, a test
przekształca się dokładnie tak jak każdą nową próbkę. Pojedyncza nowa próbka nie ma „swojej”
częstości występowania, a model oczekuje tych samych kolumn, na których go dopasowano.
Wybieranie cech na podstawie testu oznaczałoby korzystanie z próbek testowych przy
przygotowaniu danych.

## 3. Klasyfikator bazowy i k najbliższych sąsiadów

Zanim oceni się model, trzeba wiedzieć, z czym porównać jego wynik. **Klasyfikator bazowy** (ang. *baseline*, ten sam co na Wykładzie 4) zawsze wskazuje klasę liczniejszą w treningu i w ogóle nie patrzy na cechy. Model, który nie wypada lepiej od niego, niczego się z cech nie nauczył. Wynik modeli opisujemy **dokładnością** (ang. *accuracy*): odsetkiem próbek testowych, którym przypisano właściwą klasę.

Pierwszym prawdziwym modelem jest klasyfikator **k najbliższych sąsiadów** (ang. *k-nearest neighbours*, k-NN). Dla próbki testowej znajduje k najbliższych próbek treningowych (odległość euklidesowa, jak na Ćwiczeniach 2) i przypisuje jej klasę, która przeważa wśród tych sąsiadów. Bierzemy k = 5. k-NN nie ma innych parametrów dopasowywanych do danych: „uczenie” polega na zapamiętaniu próbek treningowych.

Funkcja `make_pipeline()` łączy standaryzację z sekcji 2 i model w jeden obiekt, czyli potok. `.fit()` wyznacza parametry standaryzacji na treningu i zapamiętuje próbki treningowe, a `.predict()` i `.score()` same przekształcają test parametrami z treningu, więc nie trzeba ręcznie przekształcać testu. Jako dane podajemy `Z_tr` i `Z_te` (po filtrze i CLR, przed standaryzacją).

In [ ]:
# DummyClassifier(strategy="most_frequent"): zawsze klasa najczęstsza w treningu
base = DummyClassifier(strategy="most_frequent").fit(Z_tr, y_tr)
acc_base = base.score(Z_te, y_te)        # .score(): dokładność na podanym zbiorze
print(round(acc_base, 3))   # dokładność klasyfikatora bazowego

# potok: standaryzacja, potem k-NN z n_neighbors=5 sąsiadami; .fit() dopasowuje oba kroki na treningu
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(Z_tr, y_tr)
pred_knn = knn.predict(Z_te)             # standaryzacja testu parametrami z treningu, potem klasa każdej próbki
acc_knn = knn.score(Z_te, y_te)          # dokładność k-NN: odsetek zgodnych klas
print(round(acc_knn, 3))   # dokładność k-NN
# które klasy model myli: wiersze = przewidziane, kolumny = prawdziwe; .loc: kolejność control, T2D
table = pd.crosstab(pred_knn, y_te.to_numpy(), rownames=["predicted"], colnames=["true"])
print(table.loc[["control", "T2D"], ["control", "T2D"]])

**Na co patrzeć.** Dokładność k-NN porównuje się z dokładnością klasyfikatora bazowego, a nie z zerem. Tabela pokazuje, w ilu próbkach testowych model wskazał każdą klasę i ile z nich trafił; przy około 85 próbkach jedna pomyłka więcej lub mniej zmienia dokładność o około 0,01.

🔧 **Do sprawdzenia.** Usunąć z `make_pipeline()` krok `StandardScaler()` (model dostaje wtedy cechy bez standaryzacji) albo zmienić `n_neighbors=5` na `n_neighbors=25` i uruchomić komórkę ponownie. Jak zmienia się dokładność? Na koniec przywrócić komórkę do pierwotnej postaci.

> **✏️ Zadanie 3.** Dopasować k-NN z `n_neighbors=1` (skopiować linijkę z `make_pipeline()` i nazwać model `knn1`). Policzyć jego dokładność na teście i na treningu. Skąd różnica?

In [ ]:
# potok: standaryzacja, potem k-NN z jednym sąsiadem; .fit() na treningu
knn1 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)).fit(Z_tr, y_tr)
print(round(knn1.score(Z_te, y_te), 3))   # dokładność na teście
print(knn1.score(Z_tr, y_tr))             # dokładność na treningu

**Rozwiązanie.** Na treningu k-NN z jednym sąsiadem ma dokładność 1: najbliższą próbką
treningową każdej próbki treningowej jest ona sama, z własną klasą, więc model zawsze trafia.
Na teście dokładność wynosi 0,581, czyli niewiele więcej niż klasyfikator bazowy (0,500).
Dokładność na treningu mówi tylko, że model zapamiętał próbki, na których go dopasowano, a nie
jak poradzi sobie z nowymi. Do oceny modelu nadaje się wyłącznie wynik na próbkach, których
model nie widział.

## 4. Jak bardzo wynik zależy od podziału

Dokładność z sekcji 3 pochodzi z jednego podziału i jednego zbioru testowego. Inny podział dałby inne próbki testowe, a więc inną liczbę. Żeby sprawdzić, jak duże są te różnice, powtarzamy cały cykl z sekcji 1–3 (podział, filtr, CLR, oba modele) na 20 podziałach warstwowych. Robimy to w trzech krokach.

**Krok 1: 20 podziałów naraz.** `StratifiedShuffleSplit(n_splits=20, ...)` losuje 20 podziałów warstwowych, a `.split(X, y)` podaje je kolejno. `list()` zbiera je w listę: każdy element to para (numery próbek treningowych, numery próbek testowych). Element listy wybiera się nawiasem kwadratowym, np. `splits[0]` (w Pythonie liczy się od zera).

In [ ]:
# 20 podziałów warstwowych, po 25% próbek w teście; random_state=42: ziarno
splits = list(StratifiedShuffleSplit(n_splits=20, test_size=0.25, random_state=42).split(X, y))
print(len(splits))                       # liczba elementów listy
tr, te = splits[0]                       # pierwszy podział: numery treningu i numery testu
print(len(tr), len(te))                  # liczba próbek treningowych i testowych

**Krok 2: cały cykl jako funkcja.** **Funkcja** to nazwany fragment kodu, który można wywoływać wiele razy z różnymi danymi. `def run_cycle(tr, te):` oznacza, że przy wywołaniu podaje się numery próbek treningowych i testowych. Wcięte linijki to te same kroki co w sekcjach 1–3; `X.iloc[tr]` wybiera wiersze o podanych numerach. `return` zwraca wynik funkcji. Najpierw sprawdzamy funkcję na jednym podziale. Pierwszy z 20 podziałów powstał z tym samym ziarnem 42 co podział z sekcji 1, więc jest z nim identyczny: funkcja powinna zwrócić te same dokładności co komórka z modelami.

In [ ]:
def run_cycle(tr, te):                   # tr, te: numery próbek treningowych i testowych jednego podziału
    Xa, Xb, ya, yb = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]   # trening i test (sekcja 1)
    cols = Xa.columns[(Xa > 0).mean() >= 0.10]                   # filtr wyznaczony na treningu (sekcja 2)
    Za, Zb = clr(Xa[cols]), clr(Xb[cols])                        # CLR treningu i testu
    base = DummyClassifier(strategy="most_frequent").fit(Za, ya)  # klasyfikator bazowy (sekcja 3)
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(Za, ya)   # potok: standaryzacja + k-NN
    return {"baseline": base.score(Zb, yb), "knn": knn.score(Zb, yb)}   # wynik: dokładność obu modeli na teście

print(run_cycle(tr, te))                 # jedno wywołanie: te same liczby co acc_base i acc_knn z sekcji 3

**Krok 3: funkcja dla każdego podziału.** Pętla `for tr, te in splits:` przechodzi po kolei przez 20 elementów listy; w każdym powtórzeniu `tr` i `te` to numery z kolejnego podziału. Wynik każdego wywołania dopisujemy do listy, a na końcu zamieniamy listę na tabelę: wiersz = podział, kolumny = modele.

In [ ]:
rows = []                                # pusta lista na wyniki
for tr, te in splits:                    # pętla po 20 podziałach
    rows.append(run_cycle(tr, te))       # dopisanie wyniku jednego podziału
results = pd.DataFrame(rows, index=range(1, 21))   # tabela 20 × 2; podziały numerowane od 1
print(results.head())                    # pierwsze wiersze wyniku
print(results.agg(["min", "max"]).round(3))   # najmniejsza i największa dokładność każdego modelu
print(results.mean().round(3))           # średnia dokładność z 20 podziałów

# jeden punkt = dokładność k-NN w jednym podziale; linia przerywana = klasyfikator bazowy
fig, ax = plt.subplots(figsize=(7, 4))   # nowa rycina z jednym panelem
ax.scatter(results.index, results["knn"], color="#2C5F7C")   # punkty k-NN
ax.axhline(results["baseline"].mean(), linestyle="--", color="black")   # średnia dokładność klasyfikatora bazowego
ax.set_xlabel("Numer podziału"); ax.set_ylabel("Dokładność k-NN na teście")   # podpisy osi
plt.show()   # wyświetlenie ryciny

**Na co patrzeć.** Klasyfikator bazowy ma prawie zawsze tę samą dokładność (0,500 albo 0,512, zależnie od tego, czy w teście są 43 czy 42 próbki T2D), bo podział warstwowy zachowuje proporcje klas. Dokładność k-NN zmienia się wyraźnie między podziałami, choć dane i model są te same; zmieniają się tylko próbki, które trafiły do testu.

> **✏️ Zadanie 4.** Przewagę k-NN nad klasyfikatorem bazowym w każdym podziale daje `results["knn"] - results["baseline"]`. W którym podziale przewaga jest najmniejsza i ilu próbkom testowym odpowiada?

In [ ]:
gain = results["knn"] - results["baseline"]   # przewaga k-NN w każdym z 20 podziałów
print(round(gain.min(), 3))             # najmniejsza przewaga
print(gain.idxmin())                    # numer podziału, w którym wystąpiła
print(round(gain.min() * len(y_te)))    # ta przewaga w liczbie próbek testowych

**Rozwiązanie.** Najmniejsza przewaga wynosi 0,023 i występuje w podziale 1 (k-NN 0,523 wobec
0,500), co odpowiada 2 próbkom testowym. Podział 1 to ten sam podział co w sekcji 3, więc wynik
z sekcji 3 był akurat najgorszym z 20. Przy najlepszym podziale ten sam model ma dokładność
0,686. Oba wnioski dotyczą tych samych danych i tego samego modelu, różnią się tylko próbki
w teście. Wynik z jednego podziału jest więc mało wiarygodny, a różnice kilku punktów
procentowych, np. między wartościami k, mieszczą się w tym rozrzucie.